# Data analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from functions import find_neighboring_directories
from make_json import parameter_names

In [ ]:
neighbouring_directories = find_neighboring_directories()
print(neighbouring_directories)
# import .npy files from neighbouring directories
data = {}
parameter_values = {}
for directory in neighbouring_directories:
    print(directory)
    for filename in os.listdir(directory):
        if filename.endswith(".npy") and filename.startswith("overall"):
            filepath = os.path.join(directory, filename)
            data[directory] = np.load(filepath)
        if filename.endswith(".npy") and filename.startswith("parameter_values"):
            filepath = os.path.join(directory, filename)
            parameter_values[directory] = np.load(filepath)

print(data.keys())
print(parameter_values)

In [ ]:
params_05 = np.load("NOISE/parameter_values_5.npy")
returns_05 = np.load("NOISE/overall_returns_05.npy")
params_1 = np.load("NOISE/parameter_values_1.npy")      
returns_1 = np.load("NOISE/overall_returns_1.npy")  
params_2 = np.load("NOISE/parameter_values_2.npy")  
returns_2 = np.load("NOISE/overall_returns_2.npy")

returns_dir = {"0.5": returns_05, "1": returns_1, "2": returns_2}
params_dir = {"0.5": params_05, "1": params_1, "2": params_2}
color_dir = {"0.5": "blue", "1": "green", "2": "red"}

def plot_results(returns_dir, params_dir):
    plt.figure(figsize=(10, 9)) 
    for temp_level in ["0.5", "1", "2"]:
        returns = np.array(returns_dir[temp_level]) * 100
        params = params_dir[temp_level]
        sorted_indices = np.argsort(params)
        sorted_params = np.array(params)[sorted_indices]
        returns = returns[:, sorted_indices]

        # Calculate the percentage of runs above the threshold
        above_threshold = np.sum(returns > 70, axis=0) / len(returns) * 100

        # Plot the results
        plt.bar(range(len(sorted_params)), above_threshold, color= color_dir[temp_level], alpha=0.1, label=f"Temp: {temp_level}")
        for i in range(len(sorted_params)):
            plt.plot(i * np.ones_like(returns[:, i]), returns[:, i], 'o', color= color_dir[temp_level], markersize=2, alpha=0.01)
            plt.text(i, above_threshold[i], f"{above_threshold[i]:.1f}%", ha='center', va='bottom', fontsize=8)

    plt.xlabel('Parameter set')
    plt.ylabel('Percentage of runs above 70%')
    plt.title(f"Variability of performance with temperature and noise")
    plt.hlines(70, -0.5, len(sorted_params) - 0.5, colors='r', linestyles='dashed', label='Threshold = 70%')
    plt.xticks(range(len(sorted_params)), labels=[f"{p:.2f}" for p in sorted_params], rotation=45)
    plt.ylim(0, 100)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_results(returns_dir, params_dir)

In [ ]:
plot_results(returns_dir, params_dir, "0.5")

In [ ]:
plot_results(returns_05, params_05, "temp = 0.05")
plot_results(returns_1, params_1, "temp = 0.1")
plot_results(returns_2, params_2, "noise = 0.2")

In [ ]:
for index, directory in enumerate(neighbouring_directories):
    if directory in data.keys():
        # if directory in ['ANNEALING', 'ANNEALING_NO']:
        param = directory
        returns = 100 * data[directory]
        n_values = len(parameter_values[directory])
        print(n_values)
        above_threshold = np.zeros(n_values)
        for i in range(n_values):
            above_threshold[i] = np.sum(returns[:, i] > 70)
        above_threshold /= returns.shape[0]
        plt.figure()
        means = np.mean(returns, axis=0)
        std = np.std(returns, axis=0)
        # Use plt.bar for means with error bars for std
        plt.bar(range(n_values), means, label='mean', alpha=0.2, width=0.5)
        for i in range(n_values):
            plt.plot(i * np.ones_like(returns[:, i]), returns[:, i], '.', alpha = 0.5, color='b')
        plt.ylim(0, 100)
        for i in range(n_values):
            plt.text(i, 71, f"{100*above_threshold[i]:.2f} %", ha='center')
        plt.hlines(70, 0, n_values - 1, colors='y', linestyles='dashed')
        plt.xticks(np.arange(n_values, step=1), labels=parameter_values[directory])
        plt.title(f'Simulated Annealing different noise values')
        plt.xlabel(param)
        plt.ylabel('Return %')
        plt.legend()
        plt.show()

In [ ]:
# def plot_returns(data, key):
for index, directory in enumerate(neighbouring_directories):
    if directory in data.keys():
        param = directory
        returns = 100*data[directory]
        n_values = len(parameter_values[directory])
        print(n_values)
        above_threshold = np.zeros(n_values)
        for i in range(n_values):
            above_threshold[i] = np.sum(returns[:,i] > 70)
        above_threshold /= returns.shape[0]
        plt.figure()
        means = np.mean(returns, axis=0)
        std = np.std(returns, axis=0)
        plt.plot(means, label='mean', marker='o')
        plt.fill_between(range(n_values), means-std, means+std, alpha=0.5)
        for i in range(n_values):
            plt.plot(i*np.ones_like(returns[:,i]), returns[:,i], 'x')
        plt.ylim(0, 100)
        for i in range(n_values):
            plt.text(i, 70, f"{above_threshold[i]:.2f}", ha='center')
        plt.hlines(70, 0, n_values - 1, colors='y', linestyles='dashed')
        plt.xticks(np.arange(n_values, step=1), labels = parameter_values[directory])
        plt.title(f'Performance for different values of {param}')
        plt.xlabel(param)
        plt.ylabel('Return %')
        plt.legend()
        plt.show()


In [ ]:
def plot_returns(data, key):
  for index, directory in enumerate(neighbouring_directories):
    if directory in data.keys():
      param = directory
      returns = 100 * data[directory]
      parameter_values_sorted = sorted(parameter_values[directory])  # Sort parameter values
      sorted_indices = np.argsort(parameter_values[directory])  # Get sorting indices

      # Arrange returns based on sorted parameter values
      returns = returns[:, sorted_indices]

      n_values = len(parameter_values_sorted)
      above_threshold = np.zeros(n_values)
      for i in range(n_values):
        above_threshold[i] = np.sum(returns[:, i] > 70)

      above_threshold /= returns.shape[0]
      plt.figure()
      means = np.mean(returns, axis=0)
      std = np.std(returns, axis=0)
      plt.plot(means, label='mean', marker='o')
      plt.fill_between(range(n_values), means - std, means + std, alpha=0.5)

      for i in range(n_values):
        plt.plot(i * np.ones_like(returns[:, i]), returns[:, i], 'x')
      plt.ylim(0, 100)

      for i in range(n_values):
        plt.text(i, 70, f"{above_threshold[i]:.2f}", ha='center')
      plt.hlines(70, 0, n_values - 1, colors='y', linestyles='dashed')
      plt.xticks(np.arange(n_values, step=1), labels=parameter_values_sorted)
      plt.title(f'Different noise levels for Simulated Annealing')
      plt.xlabel(param)
      plt.ylabel('Return %')
      plt.legend()
      plt.show()

# sort parameter_values ascending and the sort the returns in the same way
plot_returns(data, parameter_names)